Dependencies (might be incomplete)

Using conda (conda install (-c conda-forge) \*name\*):
- esgf-pyclient
- xarray
- cartopy
- matplotlib
- geopandas
- numpy
- pandas
- dask
- cftime
- rioxarray
- pydap
- pykrige
- scipy
- sklearn

In [2]:
from pyesgf.search import SearchConnection
from pyesgf.search import SearchConnection
from pyesgf.search.results import DatasetResult as DatasetResult
import xarray as xr
import cartopy.crs as ccrs
from cartopy import io
from cartopy.util import add_cyclic_point
import matplotlib.pyplot as plt
from datetime import datetime as dt
import re
import os
import geopandas as gpd
import warnings
import numpy as np
import pandas as pd
from dask.distributed import Client, LocalCluster
import dask
from tools import result_to_dict, load_datasets, get_distance, clip, get_obs_pr_data
import logging
import dask.dataframe as dd
from calendar import monthrange
import pykrige as pk
from sklearn.linear_model import LinearRegression
import scipy.stats as stats

In [3]:
# Example nodes. Due to outages, you may need to change the address in SearchConnection.
# Also note that sometimes (quite often), there will be problems with using ESGF this way.
# One way to circumvent the problems is to download the actual files manually. If this way is used, you can just
# from tools import process
# ds = xr.open_mfdataset([ds_files],preprocess=lambda ds: process(ds,var,min_date,max_date))
# to make sure that the format is correct
# Then simply add them do a dictionary (dsets[model_name]=dset) for each

#https://esgf-node.llnl.gov/esg-search
#https://esgf.ceda.ac.uk/esg-search
#https://esgf-data.dkrz.de/esg-search #does not inlude all models

In [4]:
# Connect to ESGF
conn = SearchConnection('https://esgf.ceda.ac.uk/esg-search', distrib=True)
# Turn off warnings (generally not advisible but here it makes sense)
os.environ["ESGF_PYCLIENT_NO_FACETS_STAR_WARNING"] = "0"
warnings.simplefilter("ignore")
logging.getLogger('distributed').setLevel(logging.WARNING)

In [5]:
# Temporal limits of the data
min_date = dt(1980,1,1)
max_date = dt(2024,12,31)
# Parameters for seaching
var = "pr"
exp = "historical"
tbl = "Amon"
variant = "r1i1p1f1"
source_id = ["ACCESS-CM2",
            "MIROC6",
            "CESM2-WACCM-FV2",
            "EC-Earth3-AerChem",
            "GISS-E2-2-H",
            "NorESM2-MM"]


# Search for matches
ctx = conn.new_context(
table_id = tbl,
experiment_id = exp,
variant_label = variant,
variable_id = var,
source_id = source_id,
latest = "true"
)
# Check if matches found
if ctx.hit_count == 0:
    print("No datasets found")
    raise SystemExit()
else:
    # If yes, print number of matches and get matches
    print(ctx.hit_count)
    result = ctx.search()

30


In [6]:
# Downloads the required shapefile (if not already downloaded)
fname = io.shapereader.natural_earth(
    resolution='10m', 
    category='cultural',
    name='admin_0_countries')
geodf = gpd.read_file(fname,crs="epsg:4326")
# Shapefile and CMIP6 should both have the same coordinate reference system (crs)
crs = geodf.crs

In [7]:
result_dict_model = result_to_dict(result)

In [8]:
res = {key:load_datasets((key,value),var,min_date,max_date) for key,value in result_dict_model.items()}
del result_dict_model

In [9]:
# Excecute loading. Cluster parameters can be altered to fit the system.
cluster = LocalCluster(n_workers=6,threads_per_worker=1,memory_limit="2GiB")
client = Client(cluster)
print(client.dashboard_link)
dsets = dask.compute(res)[0]
del res
client.close()
cluster.close()
del client
del cluster

http://127.0.0.1:8787/status


Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)Error:
Error:curl error: Problem with the SSL CA cert (path? access rights?)curl error details: 
curl error: Problem with the SSL CA cert (path? access rights?)

oc_open: Could not read url
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Note:Caching=1
Note:Caching=1
Note:Caching=1
Note:Caching=1
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Note:Caching=1
Error:cu

KilledWorker: Attempted to run task 'load_datasets-3caf7927-b324-4aa7-b769-b83b8fc585c9' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:37059. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
for model, value in dsets.items():
    # Some models deviate from the standars date, this should fix that
    if pd.to_datetime(value["time"].time.values[0]).day == 15:
        dsets[model]["time"] = value["time"]+np.timedelta64(1,"D")
    dsets[model]["time"] = pd.DatetimeIndex(dsets[model]["time"])

Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Note:Caching=1
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Note:Caching=1
Error:curl error: Problem with the SSL CA cert (path? access rights?)
curl error details: 
Note:Caching=1
Note:Caching=1
Note:Caching=1
Note:Caching=1


In [ ]:
###OBSERVATIONS###

In [ ]:
meta_url = "https://www.ncei.noaa.gov/data/ghcnm/v4/precipitation/doc/ghcn-m_v4_prcp_inventory.txt"
data_url = "https://www.ncei.noaa.gov/data/ghcnm/v4/precipitation/access/"
# Load observation data
# Sometimes the code gets stucks so that one task isn't completed (check the link below)
# In these cases, interrupt the excecution and a second try is automatically started
# Usually this cell should take less than a minute so at ~ two minutes it might be worth it to interrupt
df = get_obs_pr_data(meta_url,data_url)

In [ ]:
# Metadata for output
meta = dict(zip(["latitude","longitude","elevation","time","pr","month","year"],[float,float,float,"datetime64[ns]",float,int,int]))

def manipulate(data):
    # Format date
    data["4 digit year and 2 digit month"] = dd.to_datetime(data["4 digit year and 2 digit month"].astype("str")+"-01",format="%Y%m-%d")
    # Rename columns
    data = data.rename({"precipitation value":"pr","4 digit year and 2 digit month":"time"},axis=1)
    # Limit temporally to the historical CMIP6 era
    data = data.loc[(data["time"]>="1980-01-01") & (data["time"]<="2014-12-31")]
    # Drop unnecessary columns
    data = data.drop(["GHCN identifier", "station name", "measurement flag", "quality control flag", "source flag", "source index"],axis=1)
    # Convert trace values (<0.1 mm) to 0 mm
    data["pr"] = data["pr"].where(data["pr"]!=-1,0)
    # Monthly record is 9300 mm so values above this are surely not correct
    data = data[data["pr"]<=93000]
    # Tenths of millimeters to millimeters
    data["pr"] /= 10
    # Split time column into year and month for interpolation purposes
    data["month"] = data["time"].dt.month
    data["year"] = data["time"].dt.year
    return data.reset_index(drop=True)

# Read all fixed-width files directly into a Dask DataFrame
ddf = dd.from_pandas(df)

# Map data manipulation
ddf = ddf.map_partitions(lambda df: manipulate(df),meta=meta)

# Compute the result
df_limited = ddf.compute(scheduler='processes')
del ddf
del df

In [ ]:
# New grid for interpolated observations
X, Y = np.mgrid[np.min(df_limited["longitude"])-1.0:np.max(df_limited["longitude"])+1.0:1.0,
            np.min(df_limited["latitude"])-1.0:np.max(df_limited["latitude"])+1.0:1.0]

In [ ]:
# Two sources for elevation data
elevation_urls = ["https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO1_Bed_g_gmt4.nc","https://pae-paha.pacioos.hawaii.edu/thredds/dodsC/srtm30plus_v11_land"]

In [ ]:
cluster = LocalCluster(n_workers=30)
client = Client(cluster)
print(client.dashboard_link)
try:
    # Primary source (quicker)
    ds = xr.open_dataset(elevation_urls[0],chunks={"lon":500,"lat":500})
    elev = ds.z.sel({"lon":np.unique(X),"lat":np.unique(Y)},method="nearest").load()
except:
    print("Primary source failed, resorting to the secondary source")
    # Back-up source
    ds = xr.open_dataset(elevation_urls[1],chunks={"lat":1000,"lon":1000})
    elev = ds.elev.sel({"lon":np.unique(X),"lat":np.unique(Y)},method="nearest").load()
client.close()
cluster.close()

In [ ]:
# Convert the DataArray to a DataFrame
elev_df = elev.to_pandas().reset_index().melt(id_vars=['lat'], var_name='lon', value_name='elevation').sort_values(by="lat").rename({"lat":"latitude","lon":"longitude"},axis=1)
elev_df["longitude"] = elev_df["longitude"].astype(np.float64)
elev_df = elev_df.fillna(0)
del elev

In [ ]:
# Distance to ocean from each point
df_limited = get_distance(df_limited)
elev_df = get_distance(elev_df)

In [ ]:
# Perform ordinary kriging as interpolation
from sklearn.linear_model import LinearRegression
import scipy.stats as stats

times = np.unique(df_limited["time"])
shape = (elev_df["latitude"].unique().shape[0],elev_df["longitude"].unique().shape[0])
krige_result = np.zeros(shape+(len(times),))
regressor = LinearRegression(n_jobs=-1,fit_intercept=True).fit(df_limited[["year","month","longitude","latitude","elevation","distance"]],df_limited["tas"])
regressor.n_jobs = 1
df_limited["detrend"] = df_limited["tas"]-regressor.predict(df_limited[["year","month","longitude","latitude","elevation","distance"]])

@dask.delayed
def krige(data,new_data,regressor):
    # krige using the detrended data
    ok = pk.ok.OrdinaryKriging(
    data['longitude'], data['latitude'], data['detrend'],
    variogram_model='spherical', coordinates_type="geographic",
    verbose=False, enable_plotting=False,nlags=15)
    predicted, ss = ok.execute('points', new_data["longitude"],new_data["latitude"])
    predicted = predicted
    lin_predicted = regressor.predict(new_data[["year","month","longitude","latitude","elevation","distance"]])
    ss = ss.data
    # kriged + linear prediction, kriging standard deviation
    return predicted+lin_predicted,np.sqrt(ss)

krige_tasks = []

for i, time in enumerate(times):
    data_time = df_limited[df_limited["time"]==time]
    new_data_time = elev_df.copy()
    new_data_time["time"] = time
    krige_tasks.append(krige(data_time,new_data_time,regressor))

cluster = LocalCluster(n_workers=6,threads_per_worker=1,memory_limit="1.5GiB")
client = Client(cluster)
print(client.dashboard_link)
kriged = dask.compute(*krige_tasks)
client.close()
cluster.close()
del client
del cluster

In [ ]:
std_result = np.zeros(shape+(len(times),))

In [ ]:
for idx, result in enumerate(kriged):
    pred = pd.DataFrame({"lon":elev_df["longitude"],"lat":elev_df["latitude"],"val":result[0],"elev":elev_df["elevation"]})
    pred = pred.sort_values(["lat","lon"])
    pred.loc[pred["elev"].isna(),"val"] = np.nan
    prediction = pred["val"].values.reshape(shape)
    krige_result[:,:,idx] = prediction

    std = pd.DataFrame({"lon":elev_df["longitude"],"lat":elev_df["latitude"],"val":result[1],"elev":elev_df["elevation"]})
    std = std.sort_values(["lat","lon"])
    std.loc[std["elev"].isna(),"val"] = np.nan
    deviation = var["val"].values.reshape(shape)
    std_result[:,:,idx] = deviation

In [ ]:
# Convert the DataFrame into a Dataset
obs_ds = xr.Dataset(data_vars={"pr":(["lat","lon","time"],krige_result),"std":(["lat","lon","time"],std_result)},coords={"time":(["time"],np.unique(df_limited["time"])),"lon":(["lon"],np.unique(elev_df["longitude"])),"lat":(["lat"],np.unique(elev_df["latitude"]))})

In [ ]:
country_list = []
for country in geodf[geodf["CONTINENT"]=="Africa"]["NAME"]:
    # Extract geometry
    country_geom = geodf[geodf["NAME"]==country].geometry
    country_list.append(country_geom)
from shapely.ops import unary_union
# Combine MultiPolygons to create a larger area
area_geometries = unary_union(country_list)

In [ ]:
clipped_models = {key: clip(value,area_geometries,crs) for key,value in dsets.items()}
clipped_models["obs"] = clip(obs_ds,area_geometries,crs)

In [ ]:
cluster = LocalCluster(n_workers=2,threads_per_worker=1,memory_limit="3GiB")
client = Client(cluster)
print(client.dashboard_link)
clipped_models = dask.compute(clipped_models)[0]
client.close()
cluster.close()

In [ ]:
def to_monthly(data):
    #print(data)
    """Convert kg/s/m2 to mm/month taking into considering the variability of month lengths"""
    lengths = []
    try:
        for time in data.time:
            lengths.append(monthrange(time.values.item().year,time.values.item().month)[1])
    except AttributeError:
        # observations are already in correct format
        # since observation are in datatime64[ns], an attribute error is raised
        return data
    return data*lengths*60*60*24

In [ ]:
timeseries = {key:to_monthly(value.mean(["lat","lon"])) for key,value in clipped_models.items()}

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
keys = list(timeseries.keys())

In [ ]:
stationary0 = seasonal_decompose(timeseries[keys[0]]["pr"].values,period=12)
stationary1 = seasonal_decompose(timeseries[keys[1]]["pr"].values,period=12)
stationary2 = seasonal_decompose(timeseries[keys[2]]["pr"].values,period=12)
stationary3 = seasonal_decompose(timeseries[keys[3]]["pr"].values,period=12)
stationary4 = seasonal_decompose(timeseries[keys[4]]["pr"].values,period=12)
stationary5 = seasonal_decompose(timeseries[keys[5]]["pr"].values,period=12)
stationary6 = seasonal_decompose(timeseries[keys[6]]["pr"].values,period=12) # observations
stationary7 = seasonal_decompose(timeseries[keys[6]]["std"].values,period=12) # std

In [ ]:
fig, axis = plt.subplots(2,3,figsize=(20,10))
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary0.trend+stationary0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary6.trend+stationary6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary1.trend+stationary1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary6.trend+stationary6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary2.trend+stationary2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary6.trend+stationary6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary3.trend+stationary3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary6.trend+stationary6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary4.trend+stationary4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary6.trend+stationary6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary5.trend+stationary5.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary6.trend+stationary6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,stationary6.trend+stationary6.resid+(stationary7.trend+stationary7.resid),stationary6.trend+stationary6.resid-(stationary7.trend+stationary7.res),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")

fig.suptitle("Mothly precipitation over Africa")

In [ ]:
ghana = geodf[geodf["NAME"]=="Ghana"]["geometry"]
ghana_tasks = {key:clip(value,ghana,crs) for key,value in clipped_models.items()}
ghana_clipped = dask.compute(ghana_tasks)[0]


timeseries_ghana = {key:to_monthly(value.pr.mean(["lat","lon"])) for key,value in ghana_clipped.items()}

stationary_ghana0 = seasonal_decompose(timeseries_ghana[keys[0]]["pr"].values,period=12)
stationary_ghana1 = seasonal_decompose(timeseries_ghana[keys[1]]["pr"].values,period=12)
stationary_ghana2 = seasonal_decompose(timeseries_ghana[keys[2]]["pr"].values,period=12)
stationary_ghana3 = seasonal_decompose(timeseries_ghana[keys[3]]["pr"].values,period=12)
stationary_ghana4 = seasonal_decompose(timeseries_ghana[keys[4]]["pr"].values,period=12)
stationary_ghana5 = seasonal_decompose(timeseries_ghana[keys[5]]["pr"].values,period=12)
stationary_ghana6 = seasonal_decompose(timeseries_ghana[keys[6]]["pr"].values,period=12) # observations
stationary_ghana7 = seasonal_decompose(timeseries_ghana[keys[6]]["std"].values,period=12) # std

fig, axis = plt.subplots(2,3,figsize=(20,10))
keys = list(timeseries_ghana.keys())
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary_ghana0.trend+stationary_ghana0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary_ghana1.trend+stationary_ghana1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary_ghana2.trend+stationary_ghana2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary_ghana3.trend+stationary_ghana3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary_ghana4.trend+stationary_ghana4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary_ghana5.trend+stationary_ghana5.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary_ghana6.trend+stationary_ghana6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,(stationary_ghana6.trend+stationary_ghana6.resid)+(stationary_ghana7.trend+stationary_ghana7.resid),(stationary_ghana6.trend+stationary_ghana6.resid)-(stationary_ghana7.trend+stationary_ghana7.resid),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")

fig.suptitle("Mothly precipitation over Ghana")

In [ ]:
####KENYA####
kenya = geodf[geodf["NAME"]=="Kenya"]["geometry"]
kenya_tasks = {key:clip(value,kenya,crs) for key,value in clipped_models.items()}
kenya_clipped = dask.compute(kenya_tasks)[0]


timeseries_kenya = {key:to_monthly(value.pr.mean(["lat","lon"])) for key,value in kenya_clipped.items()}

stationary_kenya0 = seasonal_decompose(timeseries_kenya[keys[0]]["pr"].values,period=12)
stationary_kenya1 = seasonal_decompose(timeseries_kenya[keys[1]]["pr"].values,period=12)
stationary_kenya2 = seasonal_decompose(timeseries_kenya[keys[2]]["pr"].values,period=12)
stationary_kenya3 = seasonal_decompose(timeseries_kenya[keys[3]]["pr"].values,period=12)
stationary_kenya4 = seasonal_decompose(timeseries_kenya[keys[4]]["pr"].values,period=12)
stationary_kenya5 = seasonal_decompose(timeseries_kenya[keys[5]]["pr"].values,period=12)
stationary_kenya6 = seasonal_decompose(timeseries_kenya[keys[6]]["pr"].values,period=12) # observations
stationary_kenya7 = seasonal_decompose(timeseries_kenya[keys[6]]["tas"].values,period=12) # std

fig, axis = plt.subplots(2,3,figsize=(20,10))
keys = list(timeseries_kenya.keys())
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary_kenya0.trend+stationary_kenya0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary_kenya1.trend+stationary_kenya1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary_kenya2.trend+stationary_kenya2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary_kenya3.trend+stationary_kenya3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary_kenya4.trend+stationary_kenya4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary_kenya5.trend+stationary_kenya5.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary_kenya6.trend+stationary_kenya6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,(stationary_kenya6.trend+stationary_kenya6.resid)+(stationary_kenya7.trend+stationary_kenya7.resid),(stationary_kenya6.trend+stationary_kenya6.resid)-(stationary_kenya7.trend+stationary_kenya7.resid),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")
fig.suptitle("Mothly precipitation over Kenya")

In [ ]:
####TANZANIA####
tanzania = geodf[geodf["NAME"]=="Tanzania"]["geometry"]
tanzania_tasks = {key:clip(value,tanzania,crs) for key,value in clipped_models.items()}
tanzania_clipped = dask.compute(tanzania_tasks)[0]


timeseries_tanzania = {key:to_monthly(value.pr.mean(["lat","lon"])) for key,value in tanzania_clipped.items()}

stationary_tanzania0 = seasonal_decompose(timeseries_tanzania[keys[0]]["pr"].values,period=12)
stationary_tanzania1 = seasonal_decompose(timeseries_tanzania[keys[1]]["pr"].values,period=12)
stationary_tanzania2 = seasonal_decompose(timeseries_tanzania[keys[2]]["pr"].values,period=12)
stationary_tanzania3 = seasonal_decompose(timeseries_tanzania[keys[3]]["pr"].values,period=12)
stationary_tanzania4 = seasonal_decompose(timeseries_tanzania[keys[4]]["pr"].values,period=12)
stationary_tanzania5 = seasonal_decompose(timeseries_tanzania[keys[5]]["pr"].values,period=12)
stationary_tanzania6 = seasonal_decompose(timeseries_tanzania[keys[6]]["pr"].values,period=12) # observations
stationary_tanzania7 = seasonal_decompose(timeseries_tanzania[keys[6]]["std"].values,period=12) #std

fig, axis = plt.subplots(2,3,figsize=(20,10))
keys = list(timeseries_tanzania.keys())
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary_tanzania0.trend+stationary_tanzania0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary_tanzania1.trend+stationary_tanzania1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary_tanzania2.trend+stationary_tanzania2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary_tanzania3.trend+stationary_tanzania3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary_tanzania4.trend+stationary_tanzania4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary_tanzania0.trend+stationary_tanzania0.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary_tanzania6.trend+stationary_tanzania6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,(stationary_tanzania6.trend+stationary_tanzania6.resid)+(stationary_tanzania7.trend+stationary_tanzania7.resid),(stationary_tanzania6.trend+stationary_tanzania6.resid)-(stationary_tanzania7.trend+stationary_tanzania7.resid),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")

fig.suptitle("Mothly precipitation over Tanzania")

In [ ]:
####SOUTH AFRICA####
south_africa = geodf[geodf["NAME"]=="South Africa"]["geometry"]
south_africa_tasks = {key:clip(value,south_africa,crs) for key,value in clipped_models.items()}
south_africa_clipped = dask.compute(south_africa_tasks)[0]


timeseries_south_africa = {key:to_monthly(value.pr.mean(["lat","lon"])) for key,value in south_africa_clipped.items()}

stationary_south_africa0 = seasonal_decompose(timeseries_south_africa[keys[0]]["pr"].values,period=12)
stationary_south_africa1 = seasonal_decompose(timeseries_south_africa[keys[1]]["pr"].values,period=12)
stationary_south_africa2 = seasonal_decompose(timeseries_south_africa[keys[2]]["pr"].values,period=12)
stationary_south_africa3 = seasonal_decompose(timeseries_south_africa[keys[3]]["pr"].values,period=12)
stationary_south_africa4 = seasonal_decompose(timeseries_south_africa[keys[4]]["pr"].values,period=12)
stationary_south_africa5 = seasonal_decompose(timeseries_south_africa[keys[5]]["pr"].values,period=12)
stationary_south_africa6 = seasonal_decompose(timeseries_south_africa[keys[6]]["pr"].values,period=12) # observations
stationary_south_africa7 = seasonal_decompose(timeseries_south_africa[keys[6]]["std"].values,period=12) #std

fig, axis = plt.subplots(2,3,figsize=(20,10))
keys = list(timeseries_south_africa.keys())
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary_south_africa0.trend+stationary_south_africa0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary_south_africa1.trend+stationary_south_africa1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary_south_africa2.trend+stationary_south_africa2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary_south_africa3.trend+stationary_south_africa3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary_south_africa4.trend+stationary_south_africa4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary_south_africa5.trend+stationary_south_africa5.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary_south_africa6.trend+stationary_south_africa6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,(stationary_south_africa6.trend+stationary_south_africa6.resid)+(stationary_south_africa7.trend+stationary_south_africa7.resid),(stationary_south_africa6.trend+stationary_south_africa6.resid)-(stationary_south_africa7.trend+stationary_south_africa7.resid),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")

fig.suptitle("Mothly precipitation over South Africa")

In [ ]:
####MADAGASCAR####
madagascar = geodf[geodf["NAME"]=="Madagascar"]["geometry"]
madagascar_tasks = {key:clip(value,madagascar,crs) for key,value in clipped_models.items()}
madagascar_clipped = dask.compute(madagascar_tasks)[0]


timeseries_madagascar = {key:to_monthly(value.pr.mean(["lat","lon"])) for key,value in madagascar_clipped.items()}

stationary_madagascar0 = seasonal_decompose(timeseries_madagascar[keys[0]]["pr"].values,period=12)
stationary_madagascar1 = seasonal_decompose(timeseries_madagascar[keys[1]]["pr"].values,period=12)
stationary_madagascar2 = seasonal_decompose(timeseries_madagascar[keys[2]]["pr"].values,period=12)
stationary_madagascar3 = seasonal_decompose(timeseries_madagascar[keys[3]]["pr"].values,period=12)
stationary_madagascar4 = seasonal_decompose(timeseries_madagascar[keys[4]]["pr"].values,period=12)
stationary_madagascar5 = seasonal_decompose(timeseries_madagascar[keys[5]]["pr"].values,period=12)
stationary_madagascar6 = seasonal_decompose(timeseries_madagascar[keys[6]]["pr"].values,period=12)
stationary_madagascar7 = seasonal_decompose(timeseries_madagascar[keys[6]]["std"].values,period=12)

fig, axis = plt.subplots(2,3,figsize=(20,10))
keys = list(timeseries_madagascar.keys())
labels = [re.findall(r"[^\.]*",key)[6] for key in keys[:-1]]

axis[0,0].set_title(labels[0])
axis[0,0].plot(timeseries[keys[0]].time,stationary_madagascar0.trend+stationary_madagascar0.resid)
axis[0,0].plot(timeseries[keys[0]].time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[0,0].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[0,0].set_ylabel(r"Precipitation (mm)")

axis[0,1].set_title(labels[1])
axis[0,1].plot(timeseries[keys[1]].time,stationary_madagascar1.trend+stationary_madagascar1.resid)
axis[0,1].plot(timeseries[keys[1]].time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[0,1].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[0,1].set_ylabel(r"Precipitation (mm)")

axis[0,2].set_title(labels[2])
axis[0,2].plot(timeseries[keys[2]].time,stationary_madagascar2.trend+stationary_madagascar2.resid)
axis[0,2].plot(timeseries[keys[2]].time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[0,2].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[0,2].set_ylabel(r"Precipitation (mm)")

axis[1,0].set_title(labels[3])
axis[1,0].plot(timeseries[keys[3]].time,stationary_madagascar3.trend+stationary_madagascar3.resid)
axis[1,0].plot(timeseries[keys[3]].time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[1,0].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[1,0].set_ylabel(r"Precipitation (mm)")

axis[1,1].set_title(labels[4])
axis[1,1].plot(timeseries[keys[4]].time,stationary_madagascar4.trend+stationary_madagascar4.resid)
axis[1,1].plot(timeseries[keys[4]].time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[1,1].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[1,1].set_ylabel(r"Precipitation (mm)")

axis[1,2].set_title(labels[5])
axis[1,2].plot(timeseries[keys[5]].time,stationary_madagascar5.trend+stationary_madagascar5.resid)
axis[1,2].plot(timeseries[keys[5]].time.time,stationary_madagascar6.trend+stationary_madagascar6.resid)
axis[1,2].fill_between(timeseries[keys[5]].time,(stationary_madagascar6.trend+stationary_madagascar6.resid)+(stationary_madagascar7.trend+stationary_madagascar7.resid),(stationary_madagascar6.trend+stationary_madagascar6.resid)-(stationary_madagascar7.trend+stationary_madagascar7.resid),color="tab:green",alpha=0.3)
axis[1,2].set_ylabel(r"Precipitation (mm)")

fig.suptitle("Mothly precipitation over Madagascar")